In [95]:
%pip install --upgrade --quiet google-cloud-aiplatform requests nest_asyncio google-cloud-logging
import nest_asyncio
nest_asyncio.apply()

In [96]:
import os
import logging
import requests
import json
import asyncio
import vertexai
from datetime import datetime
from typing import List, Dict, Any

from vertexai.preview import reasoning_engines
from google.cloud import logging as cloud_logging
from vertexai.generative_models import GenerativeModel, SafetySetting, HarmCategory, HarmBlockThreshold

# --- CONFIGURATION ---
PROJECT_ID: str = os.getenv("GOOGLE_CLOUD_PROJECT_ID", "qwiklabs-gcp-04-70cfc463a7f7")
LOCATION: str = os.getenv("GOOGLE_CLOUD_REGION", "us-central1")
GOOGLE_MAPS_API_KEY: str = os.getenv("GOOGLE_MAPS_API_KEY", "")
NWS_USER_AGENT: str = "ReadyNowApp 1.0"
STAGING_BUCKET: str = os.getenv("STAGING_BUCKET", f"gs://{PROJECT_ID}-adk-staging")

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

In [110]:
class FEMA_Assistant_App:
    def __init__(self, project_id: str, location: str):
        # 1. STORE CONFIG ONLY
        self.project_id = project_id
        self.location = location
        self.model_name = "gemini-2.0-flash-001"
        self._is_initialized = False

        # Placeholders
        self.weather_app = None
        self.route_app = None
        self.pipeline_app = None
        self.root_app = None

    def _initialize_system(self):
        """
        Initializes objects on the server (Lazy Loading).
        """
        if self._is_initialized:
            return

        print("Initializing FEMA System Components...")

        # --- CRITICAL FIX: Patch AsyncIO for Remote Runtime ---
        # This allows asyncio.run() to work even inside the container's existing loop
        import nest_asyncio
        nest_asyncio.apply()

        import vertexai
        from google.cloud import logging as cloud_logging

        # --- LATE IMPORTS ---
        from google.adk.agents import Agent, SequentialAgent
        from vertexai.agent_engines import AdkApp
        from google.adk.tools import google_search

        vertexai.init(project=self.project_id, location=self.location)

        # A. Init Logging
        try:
            self.log_client = cloud_logging.Client(project=self.project_id)
            self.cloud_logger = self.log_client.logger("fema_interactions")
        except:
            self.cloud_logger = None

        # B. Define Safety Configuration
        self.agent_safety_config = {
            "safety_settings": [
                {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_LOW_AND_ABOVE"},
                {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_LOW_AND_ABOVE"},
                {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_LOW_AND_ABOVE"},
                {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_LOW_AND_ABOVE"},
            ]
        }

        # C. Define Tool Functions
        def get_nws_weather(location: str) -> str:
            self._log_callback("TOOL", "EXEC", f"Weather: {location}")
            try:
                # 1. Geocode
                geo_resp = requests.get("https://maps.googleapis.com/maps/api/geocode/json",
                                      params={"address": location, "key": GOOGLE_MAPS_API_KEY}).json()
                if geo_resp['status'] != 'OK': return "Geocoding Error"
                lat = geo_resp['results'][0]['geometry']['location']['lat']
                lon = geo_resp['results'][0]['geometry']['location']['lng']

                # 2. NWS API
                headers = {"User-Agent": NWS_USER_AGENT}
                point_resp = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=headers).json()
                if "properties" not in point_resp: return "Outside NWS grid"

                forecast_url = point_resp["properties"]["forecast"]
                forecast = requests.get(forecast_url, headers=headers).json()
                periods = forecast["properties"]["periods"][:3]
                return " | ".join([f"{p['name']}: {p['temperature']}F, {p['shortForecast']}" for p in periods])
            except Exception as e:
                return f"Error: {e}"

        def get_google_route(origin: str, destination: str) -> str:
            self._log_callback("TOOL", "EXEC", f"Route: {origin}->{destination}")
            try:
                resp = requests.post(
                    "https://routes.googleapis.com/directions/v2:computeRoutes",
                    headers={"Content-Type": "application/json", "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY, "X-Goog-FieldMask": "routes.duration,routes.distanceMeters"},
                    json={"origin": {"address": origin}, "destination": {"address": destination}, "travelMode": "DRIVE"}
                ).json()
                if "routes" not in resp: return "No route found."
                route = resp["routes"][0]
                miles = route.get("distanceMeters", 0) / 1609.34
                return f"Distance: {miles:.2f} miles. Duration: {route.get('duration')}"
            except Exception as e:
                return f"Error: {e}"

        # D. Init Agents

        self.weather_app = AdkApp(agent=Agent(
            model=self.model_name,
            name="WeatherAgent",
            description="Provides weather forecasts.",
            instruction="You are a WeatherAgent. Use 'get_nws_weather' to provide forecasts.",
            tools=[get_nws_weather],
            generate_content_config=self.agent_safety_config
        ))

        self.route_app = AdkApp(agent=Agent(
            model=self.model_name,
            name="RouteAgent",
            description="Provides driving directions.",
            instruction="You are a RouteAgent. Use 'get_google_route' to calculate paths.",
            tools=[get_google_route],
            generate_content_config=self.agent_safety_config
        ))

        # Pipeline Agents
        search_agent = Agent(
            model=self.model_name,
            name="SearchAgent",
            description="Finds raw data.",
            instruction="You are a SearchAgent. Use 'google_search' to find facts.",
            tools=[google_search],
            generate_content_config=self.agent_safety_config
        )

        critique_agent = Agent(
            model=self.model_name,
            name="CritiqueAgent",
            description="Critiques information.",
            instruction="You are a CritiqueAgent. Critique the previous output for safety/accuracy.",
            generate_content_config=self.agent_safety_config
        )

        refine_agent = Agent(
            model=self.model_name,
            name="RefineAgent",
            description="Refines the final answer.",
            instruction="You are a RefineAgent. Refine the answer based on the critique.",
            generate_content_config=self.agent_safety_config
        )

        self.pipeline_app = AdkApp(agent=SequentialAgent(
            name="ResearchPipeline",
            description="Sequential Pipeline",
            sub_agents=[search_agent, critique_agent, refine_agent]
        ))

        # Root Agent Tools (Delegation Wrappers)
        def call_weather(location: str):
            self._log_callback("ROOT", "DELEGATE", f"WeatherAgent: {location}")
            # nest_asyncio allows us to call asyncio.run here safely
            return asyncio.run(self._stream_response(self.weather_app, f"Weather in {location}"))

        def call_route(query: str):
            self._log_callback("ROOT", "DELEGATE", f"RouteAgent: {query}")
            return asyncio.run(self._stream_response(self.route_app, query))

        def call_research(query: str):
            self._log_callback("ROOT", "DELEGATE", f"ResearchPipeline: {query}")
            return asyncio.run(self._stream_response(self.pipeline_app, query))

        self.root_app = AdkApp(agent=Agent(
            model=self.model_name,
            name="FEMARootAgent",
            description="Dispatcher",
            instruction="You are the FEMA Root Agent. Delegate to call_weather, call_route, or call_research.",
            tools=[call_weather, call_route, call_research],
            generate_content_config=self.agent_safety_config
        ))

        self._is_initialized = True
        print("Initialization Complete.")

    # --- HELPER METHODS ---

    def _log_callback(self, agent: str, event: str, payload: str):
        print(f"  [LOG] [{agent}] {event}: {payload[:100]}...")
        if self.cloud_logger:
            self.cloud_logger.log_struct({
                "timestamp": datetime.now().isoformat(),
                "agent": agent,
                "event": event,
                "payload": str(payload)
            })

    async def _stream_response(self, app, message):
        response = ""
        # Using sys_session ID for internal delegations
        async for event in app.async_stream_query(user_id="sys_session", message=message):
            if event.get('content') and event['content'].get('parts'):
                for part in event['content']['parts']:
                    if 'text' in part:
                        response += part['text']
        return response

    def _validate_input(self, text: str) -> bool:
        """
        Input Validation Guardrail with Strict Safety Settings.
        """
        # We perform local imports here to keep the method self-contained for pickling/unpickling safety
        from vertexai.generative_models import (
            GenerativeModel,
            SafetySetting,
            HarmCategory,
            HarmBlockThreshold
        )

        safety_settings = [
            SafetySetting(category=HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE),
            SafetySetting(category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE),
        ]

        model = GenerativeModel(self.model_name)
        prompt = f"Is this query relevant to FEMA/Weather/Safety/Maps? '{text}' Answer PASS or BLOCK."

        try:
            res = model.generate_content(prompt, safety_settings=safety_settings).text.strip().upper()
            return "PASS" in res
        except:
            return False

    # --- PUBLIC ENTRY POINT ---
    def query(self, user_input: str) -> str:
        # 1. LAZY INIT (Imports nest_asyncio)
        self._initialize_system()

        self._log_callback("SYSTEM", "USER_INPUT", user_input)

        # 2. Validation
        if not self._validate_input(user_input):
            self._log_callback("SYSTEM", "VALIDATION", "BLOCKED")
            return "I cannot answer that. Please ask about FEMA-related topics."

        # 3. Execution
        try:
            # Safe to call because nest_asyncio.apply() ran in _initialize_system
            return asyncio.run(self._stream_response(self.root_app, user_input))
        except Exception as e:
            return f"System Error: {e}"

In [111]:
# 1. Requirements
requirements = [
    "google-cloud-aiplatform[agent_engines,adk]",
    "requests",
    "google-cloud-logging",
    "nest_asyncio" # Explicitly required here
]

print("Deploying FEMA Assistant to Vertex AI Agent Engine...")

# 2. Deploy
remote_agent = reasoning_engines.ReasoningEngine.create(
    reasoning_engine=FEMA_Assistant_App(
        project_id=PROJECT_ID,
        location=LOCATION
    ),
    requirements=requirements,
    display_name="fema-assistant-final",
    description="FEMA Assistant"
)

print(f"Deployment Complete! Resource: {remote_agent.resource_name}")

INFO:vertexai.reasoning_engines._reasoning_engines:Using bucket qwiklabs-gcp-04-70cfc463a7f7-adk-staging


Deploying FEMA Assistant to Vertex AI Agent Engine...


INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/reasoning_engine.pkl
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/requirements.txt
INFO:vertexai.reasoning_engines._reasoning_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.reasoning_engines._reasoning_engines:Writing to gs://qwiklabs-gcp-04-70cfc463a7f7-adk-staging/reasoning_engine/dependencies.tar.gz
INFO:vertexai.reasoning_engines._reasoning_engines:Creating ReasoningEngine
INFO:vertexai.reasoning_engines._reasoning_engines:Create ReasoningEngine backing LRO: projects/59692225922/locations/us-central1/reasoningEngines/3760478201063669760/operations/3771540808457519104
INFO:vertexai.reasoning_engines._reasoning_engines:ReasoningEngine created. Resource name: projects/59692225922/locations/us-central1/reasoningEngines/3760478201063669760
INFO:vertexai.reasoning_en

Deployment Complete! Resource: projects/59692225922/locations/us-central1/reasoningEngines/3760478201063669760


In [112]:
print("\n--- TEST: Research Pipeline (Remote) ---")
try:
    response = remote_agent.query(user_input="What are the FEMA guidelines for flood safety?")
    print(f"\nFinal Response: {response}")
except Exception as e:
    print(f"Remote Execution Error: {e}")


--- TEST: Research Pipeline (Remote) ---

Final Response: FEMA guidelines for flood safety include:

*   **Stay Informed** Monitor local radio or TV stations for updated emergency information. Individuals should monitor NOAA Weather Radio and their local news for updates and directions provided by their local officials. Download the FEMA mobile app for weather alerts, safety tips, shelter information, and disaster resources.
*   **Understand Flood Warnings** A Flood Watch is issued when conditions are favorable for flooding. A Flood Warning is issued when flooding is imminent or occurring. A Flash Flood Watch is issued when conditions are favorable for flash flooding. A Flash Flood Warning is issued when flash flooding is imminent or occurring. A River Flood Watch is issued when river flooding is possible. A River Flood Warning is issued when river flooding is occurring or imminent.
*   **Do not walk or drive through flood waters** If you encounter a flooded roadway, do not attempt to